# 0. 主要对比

| 维度 | PPO | DPO | GRPO | GSPO |
|:---|:---|:---|:---|:---|
| 优势估计 | GAE 递推（依赖 Critic 模型） | 无优势函数（隐式奖励差） | 组内标准化（无 Critic） | 组内标准化（无 Critic，同 GRPO） |
| 在线/离线 | 在线 RL（需实时采样） | 完全离线（仅用已有偏好数据） | 在线 RL（每问题采 G 个回答） | 在线 RL（同 GRPO） |
| 显存需求 | ≈3× 模型大小（Policy+Critic+Ref） | ≈2× 模型大小（Policy+Ref） | ≈1.5× 模型大小（Policy+Ref） | ≈1.5× 模型大小（同 GRPO） |
| 能力上界 | 可超越数据（在线探索） | 受限于偏好数据质量 | 可超越数据（在线探索） | 可超越数据（在线探索） |


发展演进：
- PPO → DPO：去掉 Critic + 奖励模型 + 在线采样，变为监督学习 → 最简单但能力受限
- PPO → GRPO：去掉 Critic，用组内采样替代 → 显存减半，超参数减少
- GRPO → GSPO：重要性采样从 Token 级提升到序列级 → 训练更稳定，尤其利于 MoE 模型

# 1. KL 散度

| 算法 | KL 约束方式 | 具体形式 |
|:---|:---|:---|
| PPO | 显式 KL 惩罚项 + Clip | $L = \text{PPO-Clip}(\rho_t, A_t) + \beta \cdot D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$，KL 作为额外损失项加入，$\beta$ 可自适应调节 |
| DPO | 隐式 KL 编码 | KL 散度被"吸收"进了 $\log\frac{\pi_\theta}{\pi_{\text{ref}}}$ 对数概率比中，不需要显式计算 KL，但 $\beta$ 参数控制隐式约束强度 |
| GRPO | 显式 KL 惩罚项 + Token 级 Clip | 与 PPO 类似，但 Clip 作用于每个 token 的 $\rho_{i,t}$，KL 惩罚相对于冻结的 $\pi_{\text{ref}}$ |
| GSPO | 序列级 Clip 即可（可省略显式 KL） | 由于 $\rho_{\text{seq}}$ 波动范围天然小，序列级 Clip（$\epsilon \approx 3\text{e-4}$）已经足够约束，论文实验中省略了显式 KL 惩罚项 |


## 2. 重要性采样比率 $\rho$

| 算法 | $\rho$ 粒度 | 定义 | 梯度特点 |
|---|---|---|---|
| **PPO** | **Token 级** | $\rho_t=\dfrac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\mathrm{old}}}(a_t\mid s_t)}$ | 每个 token 更新幅度不同，但有 Critic 精确指导每步优势 |
| **DPO** | **无 $\rho$** | 不使用重要性采样（纯监督学习） | 梯度来自对数概率差的 sigmoid，无采样噪声 |
| **GRPO** | **Token 级** | $\rho_{i,t}=\dfrac{\pi_\theta(y_{i,t}\mid x,y_{i,<t})}{\pi_{\theta_{\mathrm{old}}}(y_{i,t}\mid x,y_{i,<t})}$ | 同一序列内 token 更新幅度可能差异巨大（如 0.4 vs 2.8），高方差 |
| **GSPO** | **序列级** | $\rho_{\mathrm{seq}}=\exp\left(\dfrac{1}{\lvert y\rvert}\sum_t\log\rho_t\right)$ | 同一序列所有 token 共享统一系数，梯度干净、一致、低噪声 |